# 02 — Preprocessing & closest-users features (Step 2)
Build the full ACC Arena tidy table on a **random sample of `N_USERS` users drawn from the whole
population** (~12k), engineer the **Team-8 X-closest-users features**, split **by user** (no leakage),
scale, and save ready-to-train arrays for every `X` in `X_VALUES`.

In [ ]:
# === Project config — Team 8: Throughput Prediction in a Dense 5G deployment (with Transfer Learning) ===
RANDOM_SEED      = 42
RESAMPLE_SECONDS = 60            # downsampling step in seconds. Lower = more samples, more time (try 30 or 15).
N_USERS          = 1500          # ACC Arena users, sampled at RANDOM from the full ~12k population (seeded).
                                 # Closest-users neighbours are searched WITHIN this sample.
N_USERS_SALT     = 3000          # Salt&Tar users: ALL of them (only ~1/3 are ever active; activity is id-biased)
X_VALUES         = [3, 5, 10]    # number of closest users to experiment with. X=0/1 excluded by design:
                                 # heavy co-location makes a single arbitrary neighbour uninformative.
BEST_X           = 3             # X used for the transfer-learning experiment (pick the best from notebook 04)
OUTLIER_PCT      = 99.0          # drop samples with throughput above this TRAIN percentile (None = keep all).
                                 # EDA (notebook 01): the ~1% extreme samples carry ~2/3 of the total variance.
ACTIVE_ONLY      = True           # regress only on ACTIVE users; idle/off have throughput ~0 by definition
MIN_TRAFFIC_TYPE = 2             # keep traffic_type >= this (2=const_rate, 3=video, 4=gaming, 5=http)

# --- data location ---
DRIVE_FOLDER_ID  = "1s1YCWyhN_Fv5sQraTVs4Rga-ATiKPRgo"   # shared Drive folder containing the zip
ZIP_NAME         = "L5GHDD_Dataset.zip"
DATA_ROOT        = "data/raw"     # dataset folders live here after unzip (local default)
PROCESSED_DIR    = "data/processed"
RESULTS_DIR      = "results"

import os, numpy as np, random
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: mount Drive and make outputs PERSIST across notebooks (no-op locally) ===
def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR   = "/content/drive/MyDrive/5g_throughput_team8"   # persistent project folder on your Drive
    PROCESSED_DIR = f"{PROJECT_DIR}/processed"                     # 02 writes here, 03/04/05 read from here
    RESULTS_DIR   = f"{PROJECT_DIR}/results"                       # models, metrics.csv, figures
    print("Outputs persist in:", PROJECT_DIR)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: locate and unzip the dataset (no-op locally) ===
if in_colab():
    import glob, zipfile, subprocess
    DATA_ROOT = "/content/L5GHDD_Dataset"
    if not os.path.isdir(DATA_ROOT):
        cands = glob.glob(f"/content/drive/MyDrive/**/{ZIP_NAME}", recursive=True)
        if not cands:                                   # fallback: download the shared folder
            subprocess.run(["pip", "-q", "install", "gdown"], check=False)
            import gdown
            gdown.download_folder(id=DRIVE_FOLDER_ID, output="/content/_dl", quiet=False, use_cookies=False)
            cands = glob.glob(f"/content/_dl/**/{ZIP_NAME}", recursive=True)
        assert cands, f"{ZIP_NAME} not found. Put it in your Drive or share the folder."
        print("Unzipping", cands[0], "...")
        with zipfile.ZipFile(cands[0]) as z:
            z.extractall(DATA_ROOT)
print("DATA_ROOT =", DATA_ROOT)

In [ ]:
# === Data loading helpers (raw "wide" format -> tidy per-user/per-timestamp table) ===
import glob, re, math
import pandas as pd
from scipy.spatial import cKDTree

def find_venue_dir(data_root, venue_key):
    """venue_key in {'acc','salt'}; robust to the zip's internal layout."""
    pat = {"acc": "*ACC*Arena*", "salt": "*Salt*Tar*"}[venue_key]
    hits = [os.path.join(dp, d) for dp, dn, _ in os.walk(data_root)
            for d in dn if glob.fnmatch.fnmatch(d, pat)]
    hits = sorted(set(hits), key=len)
    assert hits, f"venue '{venue_key}' not found under {data_root}"
    return hits[0]

def file_id_range(path):
    """User-id range covered by a CSV chunk, taken from its file name (…_<start>_<end>.csv)."""
    m = re.search(r'_(\d+)_(\d+)\.csv$', path)
    return int(m.group(1)), int(m.group(2))

def metric_files(venue_dir, subdir_glob, file_glob, user_ids=None):
    """CSV chunks of a metric, keeping only files whose user-id range intersects `user_ids`."""
    subs = glob.glob(os.path.join(venue_dir, subdir_glob))
    assert subs, f"no subdir matching {subdir_glob} in {venue_dir}"
    files = sorted(glob.glob(os.path.join(subs[0], file_glob)), key=lambda p: file_id_range(p)[0])
    if user_ids is not None:
        def overlaps(f):
            a, b = file_id_range(f)
            return any(a <= u <= b for u in user_ids)
        files = [f for f in files if overlaps(f)]
    assert files, f"no files matching {file_glob} in {subdir_glob}"
    return files

def all_user_ids(venue_dir):
    """Every user id present in the venue (derived from the throughput file names)."""
    ids = []
    for f in metric_files(venue_dir, "*Throughput*", "*.csv"):
        a, b = file_id_range(f)
        ids.extend(range(a, b + 1))
    return np.array(sorted(ids))

def build_grid(ref_file, resample_seconds):
    t = pd.read_csv(ref_file, usecols=[0]).iloc[:, 0].values.astype(float)
    return np.arange(t[0], t[-1] + 1, resample_seconds)

def _align(df, grid, tol):
    df = df[~df.index.duplicated(keep="first")]
    return df.reindex(grid, method="nearest", tolerance=tol)

def load_metric(files, value_name, grid, tol, user_ids):
    out = []
    for f in files:
        head = list(pd.read_csv(f, nrows=0).columns)
        cmap = {c: int(re.search(r'(\d+)', c).group(1)) for c in head[1:]}
        use = [head[0]] + [c for c in head[1:] if cmap[c] in user_ids]   # parse only sampled users
        df = pd.read_csv(f, usecols=use).rename(columns={head[0]: "time"}).set_index("time")
        df = df.rename(columns=cmap)
        df = _align(df, grid, tol)
        out.append(df.reset_index().melt(id_vars="time", var_name="user_id", value_name=value_name))
    return pd.concat(out, ignore_index=True)

def load_positions(files, grid, tol, user_ids):
    """Positions are interleaved blocks of (id, lat, lon, alt, mobileState); mobileState is the
    traffic_type. lat/lon are converted to local metres using ONE shared origin for the whole venue,
    so distances between users coming from different files are consistent."""
    frames = []
    for f in files:
        first = pd.read_csv(f, nrows=1).values.astype(float)
        ids_all = first[0, 1::5].astype(int)
        blocks = [k for k, u in enumerate(ids_all) if u in user_ids]
        if not blocks:
            continue
        use = [0] + [1 + 5 * k + j for k in blocks for j in range(5)]    # parse only sampled users
        df = pd.read_csv(f, usecols=use)
        t = df.iloc[:, 0].values.astype(float)
        arr = df.values.astype(float)
        ids = arr[0, 1::5].astype(int)
        vals = {"lat": arr[:, 2::5], "lon": arr[:, 3::5], "z": arr[:, 4::5], "traffic_type": arr[:, 5::5]}
        m = None
        for name, v in vals.items():
            d = pd.DataFrame(v, index=t, columns=ids)
            d.index.name = "time"
            d = _align(d, grid, tol).reset_index().melt(id_vars="time", var_name="user_id", value_name=name)
            m = d if m is None else m.merge(d, on=["time", "user_id"])
        frames.append(m)
    pos = pd.concat(frames, ignore_index=True)
    lat0, lon0 = pos.lat.mean(), pos.lon.mean()          # single origin for the whole venue
    R = 6371000.0
    pos["x"] = R * np.radians(pos.lon - lon0) * math.cos(math.radians(lat0))
    pos["y"] = R * np.radians(pos.lat - lat0)
    return pos[["time", "user_id", "x", "y", "z", "traffic_type"]]

def assemble(venue_key, n_users, resample_seconds, random_users=False):
    """Return a tidy DataFrame with one row per (user, timestamp).
    random_users=True draws n_users uniformly from the venue's FULL population (seeded, reproducible);
    otherwise the first n_users ids are used."""
    venue = find_venue_dir(DATA_ROOT, venue_key)
    if random_users:
        pop = all_user_ids(venue)
        rng = np.random.default_rng(RANDOM_SEED)
        user_ids = set(int(u) for u in rng.choice(pop, size=min(n_users, len(pop)), replace=False))
        print(f"{venue_key}: sampled {len(user_ids)} random users out of {len(pop)}")
    else:
        user_ids = set(range(n_users))
    tol = resample_seconds / 2
    mf = lambda sub, fg: metric_files(venue, sub, fg, user_ids)
    ref = mf("*Throughput*", "*.csv")[0]
    grid = build_grid(ref, resample_seconds)
    parts = [
        load_metric(mf("*Throughput*",     "*.csv"),    "throughput", grid, tol, user_ids),
        load_metric(mf("*BLER*",           "*.csv"),    "bler",       grid, tol, user_ids),
        load_metric(mf("*PRB*",            "*.csv"),    "prb",        grid, tol, user_ids),
        load_metric(mf("*RU_Association*", "*.csv"),    "ru_id",      grid, tol, user_ids),
        load_metric(mf("*SINR*",           "SINRDL_*.csv"), "sinr_dl", grid, tol, user_ids),
        load_metric(mf("*SINR*",           "SINRUL_*.csv"), "sinr_ul", grid, tol, user_ids),
        load_positions(mf("*Positions*",   "*.csv"), grid, tol, user_ids),
    ]
    df = parts[0]
    for p in parts[1:]:
        df = df.merge(p, on=["time", "user_id"], how="inner")
    df = df.dropna().reset_index(drop=True)
    df["user_id"]      = df["user_id"].astype(int)
    df["traffic_type"] = df["traffic_type"].round().astype(int)
    df["ru_id"]        = df["ru_id"].round().astype(int)
    return df

In [ ]:
# === Closest-users feature engineering (Team-8 specific) ===
# For each (timestamp, user) we append the features of its X nearest users at that
# instant (3-D Euclidean distance on x,y,z). Per neighbour: [dist, sinr_dl, sinr_ul, prb, bler, throughput].
NEIGHBOR_FEATS = ["sinr_dl", "sinr_ul", "prb", "bler", "throughput"]

def add_closest_user_features(df, x_max):
    cols = []
    for k in range(x_max):
        cols += [f"nb{k}_dist"] + [f"nb{k}_{c}" for c in NEIGHBOR_FEATS]
    out = np.full((len(df), len(cols)), np.nan)
    feat = df[NEIGHBOR_FEATS].values
    for _, idx in df.groupby("time").groups.items():
        idx = np.asarray(idx)
        if len(idx) <= 1:
            continue
        tree = cKDTree(df.loc[idx, ["x", "y", "z"]].values)
        k = min(x_max + 1, len(idx))
        dist, nb = tree.query(df.loc[idx, ["x", "y", "z"]].values, k=k)
        for r in range(len(idx)):
            picks = [j for j in range(k) if nb[r, j] != r][:x_max]
            vals = []
            for j in picks:
                vals.append(dist[r, j]); vals.extend(feat[idx[nb[r, j]]])
            out[idx[r], :len(vals)] = vals
    return pd.concat([df, pd.DataFrame(out, columns=cols, index=df.index)], axis=1)

def neighbor_cols(x):
    cols = []
    for k in range(x):
        cols += [f"nb{k}_dist"] + [f"nb{k}_{c}" for c in NEIGHBOR_FEATS]
    return cols

## Build the full tidy table and the neighbour features
We compute neighbours once at `max(X_VALUES)` and slice smaller `X` from it.

**Order matters:** the `ACTIVE_ONLY` filter is applied **after** the neighbour features, so neighbours are
still drawn from the *full* population present at each instant (idle devices still occupy space and load the
cell), while we **train/evaluate only on active users** (idle/off throughput is ~0 by definition).

In [ ]:
import time as _time
t0 = _time.time()
df = assemble("acc", n_users=N_USERS, resample_seconds=RESAMPLE_SECONDS, random_users=True)
print("tidy shape:", df.shape, "| users:", df.user_id.nunique(), "| timestamps:", df.time.nunique())
df = add_closest_user_features(df, x_max=max(X_VALUES))
print("with neighbour features:", df.shape, "| built in %.1fs" % (_time.time()-t0))

if ACTIVE_ONLY:
    before = len(df)
    df = df[df.traffic_type >= MIN_TRAFFIC_TYPE].reset_index(drop=True)
    print(f"ACTIVE_ONLY: kept {len(df)}/{before} rows (traffic_type >= {MIN_TRAFFIC_TYPE})")
print("traffic types kept:", sorted(df.traffic_type.unique()))

## Split by user (train / val / test = 70 / 15 / 15)
Splitting on users avoids leaking a user's samples across splits.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
users = df.user_id.unique(); rng.shuffle(users)
n = len(users); tr = users[:int(.7*n)]; va = users[int(.7*n):int(.85*n)]; te = users[int(.85*n):]
split = {u:"train" for u in tr}; split.update({u:"val" for u in va}); split.update({u:"test" for u in te})
df["split"] = df.user_id.map(split)
print(df.split.value_counts())

## Drop extreme-throughput outliers
As motivated in the EDA, the top ~1% samples carry most of the variance and would dominate squared-error
training and R². The threshold is computed on the **train split only** (no leakage) and applied to all splits,
so the whole study is scoped to typical operating conditions.

In [ ]:
if OUTLIER_PCT is not None:
    thr = float(np.percentile(df.loc[df.split=="train", "throughput"], OUTLIER_PCT))
    before = len(df)
    df = df[df.throughput <= thr].reset_index(drop=True)
    print(f"outlier threshold (p{OUTLIER_PCT} of train) = {thr:.2f} Mbps -> "
          f"removed {before-len(df)} rows ({(before-len(df))/before*100:.2f}%)")

## Feature matrix builder
Base features: numeric `[bler, prb, sinr_dl, sinr_ul, x, y, z]` (standardised) + one-hot `traffic_type`
(fixed 6 classes) + the selected neighbour columns. We deliberately keep a **fixed feature schema** so the
same model can be reused for transfer learning (notebook 05). `ru_id` is venue-specific and left out.

In [ ]:
from sklearn.preprocessing import StandardScaler
import json

BASE_NUM = ["bler","prb","sinr_dl","sinr_ul","x","y","z"]
TRAFFIC_CLASSES = [0,1,2,3,4,5]

def onehot_traffic(frame):
    return pd.DataFrame({f"traffic_{c}": (frame.traffic_type==c).astype(float) for c in TRAFFIC_CLASSES},
                        index=frame.index)

def build_matrix(frame, x, scaler=None, fit=False):
    num_cols = BASE_NUM + neighbor_cols(x)
    num = frame[num_cols].astype(float)
    num = num.fillna(num.median())            # neighbour gaps -> column median
    if fit:
        scaler = StandardScaler().fit(num)
    num_s = pd.DataFrame(scaler.transform(num), columns=num_cols, index=frame.index)
    X = pd.concat([num_s, onehot_traffic(frame)], axis=1)
    return X.values.astype("float32"), list(X.columns), scaler

## Save arrays for every X

In [ ]:
import joblib
tr_m, va_m, te_m = df.split=="train", df.split=="val", df.split=="test"
y = df["throughput"].values.astype("float32")
saved = []
for x in X_VALUES:
    Xtr, cols, scaler = build_matrix(df[tr_m], x, fit=True)
    Xva, _, _ = build_matrix(df[va_m], x, scaler=scaler)
    Xte, _, _ = build_matrix(df[te_m], x, scaler=scaler)
    np.savez_compressed(f"{PROCESSED_DIR}/acc_X{x}.npz",
                        X_train=Xtr, y_train=y[tr_m], X_val=Xva, y_val=y[va_m],
                        X_test=Xte, y_test=y[te_m])
    joblib.dump(scaler, f"{PROCESSED_DIR}/acc_X{x}_scaler.pkl")
    json.dump(cols, open(f"{PROCESSED_DIR}/acc_X{x}_cols.json","w"))
    saved.append((x, Xtr.shape[1]))
    print(f"X={x:>2}  features={Xtr.shape[1]:>3}  train={Xtr.shape[0]}")
print("saved:", saved)

Arrays for each `X` are now in `data/processed/`. Notebook **03** trains the models on them.